# 18. 3D geometry and rendering — complete small-tensor NeRF and 3D Gaussian Splatting paths

Only widths, ray/image counts and sample counts are reduced.

NeRF keeps the original field architecture: positional encoding, 8-layer position MLP with the layer-4 skip, view-independent density, view-dependent color branch, coarse rendering, PDF importance sampling, and a separate fine network.

3DGS keeps anisotropic 3D covariance, quaternion rotation, SH appearance, alpha compositing and the adaptive density-control operations: clone, split, prune and opacity reset.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(3)
device = torch.device("cpu")
print("device:", device)

## 1. NeRF positional encoding and original 8-layer field

In [ ]:
def positional_encoding(x, frequency_bands):
    encoded = [x]
    for frequency in frequency_bands:
        encoded.append(torch.sin(frequency * x))
        encoded.append(torch.cos(frequency * x))
    return torch.cat(encoded, dim=-1)


class NeRFField(nn.Module):
    def __init__(
        self,
        hidden_dim=32,
        position_frequencies=10,
        direction_frequencies=4,
    ):
        super().__init__()

        self.position_frequency_bands = 2.0 ** torch.arange(
            position_frequencies,
            dtype=torch.float32,
        )
        self.direction_frequency_bands = 2.0 ** torch.arange(
            direction_frequencies,
            dtype=torch.float32,
        )

        position_dim = 3 * (1 + 2 * position_frequencies)
        direction_dim = 3 * (1 + 2 * direction_frequencies)

        self.position_layers = nn.ModuleList()
        for layer_index in range(8):
            if layer_index == 0:
                input_dim = position_dim
            elif layer_index == 4:
                input_dim = hidden_dim + position_dim
            else:
                input_dim = hidden_dim

            self.position_layers.append(
                nn.Linear(input_dim, hidden_dim)
            )

        self.density_head = nn.Linear(hidden_dim, 1)
        self.feature_head = nn.Linear(hidden_dim, hidden_dim)
        self.color_hidden = nn.Linear(
            hidden_dim + direction_dim,
            hidden_dim // 2,
        )
        self.color_head = nn.Linear(hidden_dim // 2, 3)

    def forward(self, position, direction):
        position_frequencies = self.position_frequency_bands.to(
            position.device
        )
        direction_frequencies = self.direction_frequency_bands.to(
            direction.device
        )

        encoded_position = positional_encoding(
            position,
            position_frequencies,
        )
        hidden = encoded_position

        for layer_index, layer in enumerate(self.position_layers):
            if layer_index == 4:
                hidden = torch.cat(
                    [hidden, encoded_position],
                    dim=-1,
                )
            hidden = F.relu(layer(hidden))

        density = F.relu(self.density_head(hidden))
        feature = self.feature_head(hidden)

        encoded_direction = positional_encoding(
            direction,
            direction_frequencies,
        )
        color_hidden = torch.cat(
            [feature, encoded_direction],
            dim=-1,
        )
        color_hidden = F.relu(self.color_hidden(color_hidden))
        color = torch.sigmoid(self.color_head(color_hidden))
        return color, density.squeeze(-1)


coarse_field = NeRFField().to(device)
fine_field = NeRFField().to(device)

assert len(coarse_field.position_layers) == 8
assert coarse_field.position_layers[4].in_features > coarse_field.position_layers[3].out_features
assert coarse_field.density_head.out_features == 1
assert coarse_field.color_head.out_features == 3
print("NeRF position layers:", len(coarse_field.position_layers))

## 2. Volume rendering and hierarchical PDF sampling

In [ ]:
def volume_render(color, density, depths):
    delta = depths[..., 1:] - depths[..., :-1]
    last_delta = torch.full_like(depths[..., -1:], 1e10)
    delta = torch.cat([delta, last_delta], dim=-1)

    alpha = 1 - torch.exp(-density * delta)
    transmittance = torch.cumprod(
        torch.cat(
            [
                torch.ones_like(alpha[..., :1]),
                1 - alpha + 1e-10,
            ],
            dim=-1,
        ),
        dim=-1,
    )[..., :-1]
    weights = transmittance * alpha
    rgb = (weights[..., None] * color).sum(dim=-2)
    return rgb, weights


def sample_pdf(bins, weights, sample_count):
    weights = weights + 1e-5
    pdf = weights / weights.sum(dim=-1, keepdim=True)
    cdf = torch.cumsum(pdf, dim=-1)
    cdf = torch.cat(
        [torch.zeros_like(cdf[..., :1]), cdf],
        dim=-1,
    )

    u = torch.linspace(
        0.0,
        1.0,
        sample_count,
        device=bins.device,
    )
    u = u.expand(*cdf.shape[:-1], sample_count).contiguous()

    indices = torch.searchsorted(cdf.contiguous(), u, right=True)
    below = (indices - 1).clamp_min(0)
    above = indices.clamp_max(cdf.size(-1) - 1)

    cdf_pair = torch.stack(
        [
            torch.gather(cdf, -1, below),
            torch.gather(cdf, -1, above),
        ],
        dim=-1,
    )
    bin_pair = torch.stack(
        [
            torch.gather(bins, -1, below),
            torch.gather(bins, -1, above),
        ],
        dim=-1,
    )

    denominator = cdf_pair[..., 1] - cdf_pair[..., 0]
    denominator = torch.where(
        denominator < 1e-5,
        torch.ones_like(denominator),
        denominator,
    )
    interpolation = (
        u - cdf_pair[..., 0]
    ) / denominator
    return (
        bin_pair[..., 0]
        + interpolation
        * (bin_pair[..., 1] - bin_pair[..., 0])
    )


def query_nerf(field, ray_origin, ray_direction, depths):
    points = (
        ray_origin[:, None, :]
        + ray_direction[:, None, :] * depths[..., None]
    )
    directions = ray_direction[:, None, :].expand_as(points)
    color, density = field(
        points.reshape(-1, 3),
        directions.reshape(-1, 3),
    )
    color = color.view(points.size(0), points.size(1), 3)
    density = density.view(points.size(0), points.size(1))
    return color, density


def hierarchical_nerf_render(
    coarse_field,
    fine_field,
    ray_origin,
    ray_direction,
    coarse_samples=8,
    fine_samples=8,
    near=0.5,
    far=3.0,
):
    # Sample counts are reduced tensor lengths; both coarse/fine stages remain.
    coarse_depths = torch.linspace(
        near,
        far,
        coarse_samples,
        device=ray_origin.device,
    ).expand(ray_origin.size(0), -1)

    coarse_color, coarse_density = query_nerf(
        coarse_field,
        ray_origin,
        ray_direction,
        coarse_depths,
    )
    coarse_rgb, coarse_weights = volume_render(
        coarse_color,
        coarse_density,
        coarse_depths,
    )

    midpoints = 0.5 * (
        coarse_depths[..., 1:]
        + coarse_depths[..., :-1]
    )
    fine_depths = sample_pdf(
        midpoints,
        coarse_weights[..., 1:-1].detach(),
        fine_samples,
    )

    all_depths = torch.sort(
        torch.cat([coarse_depths, fine_depths], dim=-1),
        dim=-1,
    ).values
    fine_color, fine_density = query_nerf(
        fine_field,
        ray_origin,
        ray_direction,
        all_depths,
    )
    fine_rgb, fine_weights = volume_render(
        fine_color,
        fine_density,
        all_depths,
    )
    return coarse_rgb, fine_rgb, coarse_weights, fine_weights


ray_origin = torch.zeros(4, 3, device=device)
ray_direction = F.normalize(torch.randn(4, 3, device=device), dim=-1)
coarse_rgb, fine_rgb, coarse_weights, fine_weights = hierarchical_nerf_render(
    coarse_field,
    fine_field,
    ray_origin,
    ray_direction,
)
loss = fine_rgb.square().mean() + coarse_rgb.square().mean()
loss.backward()

assert coarse_weights.size(-1) == 8
assert fine_weights.size(-1) == 16
print("coarse/fine RGB:", coarse_rgb.shape, fine_rgb.shape)

## 3. 3DGS parameterization: log-scale + quaternion covariance + SH

In [ ]:
def quaternion_to_rotation(quaternion):
    quaternion = F.normalize(quaternion, dim=-1)
    w, x, y, z = quaternion.unbind(dim=-1)

    rotation = torch.stack(
        [
            1 - 2 * (y.square() + z.square()),
            2 * (x * y - w * z),
            2 * (x * z + w * y),
            2 * (x * y + w * z),
            1 - 2 * (x.square() + z.square()),
            2 * (y * z - w * x),
            2 * (x * z - w * y),
            2 * (y * z + w * x),
            1 - 2 * (x.square() + y.square()),
        ],
        dim=-1,
    )
    return rotation.view(-1, 3, 3)


def covariance_3d(log_scales, quaternion):
    scales = torch.exp(log_scales)
    rotation = quaternion_to_rotation(quaternion)
    linear = rotation @ torch.diag_embed(scales)
    return linear @ linear.transpose(-1, -2)


def sh_degree1(direction):
    direction = F.normalize(direction, dim=-1)
    x, y, z = direction.unbind(dim=-1)
    c0 = 0.2820947918
    c1 = 0.4886025119
    return torch.stack(
        [
            torch.full_like(x, c0),
            -c1 * y,
            c1 * z,
            -c1 * x,
        ],
        dim=-1,
    )


def evaluate_sh(coefficients, direction):
    basis = sh_degree1(direction)
    return torch.sigmoid(
        torch.einsum("nk,nkc->nc", basis, coefficients)
    )

## 4. Differentiable Gaussian rasterizer and front-to-back compositing

In [ ]:
def project_gaussians(
    means_3d,
    covariance_values,
    height,
    width,
    fx=20.0,
    fy=20.0,
):
    depth = means_3d[:, 2].clamp_min(0.2)
    u = fx * means_3d[:, 0] / depth + width / 2
    v = fy * means_3d[:, 1] / depth + height / 2
    means_2d = torch.stack([u, v], dim=-1)
    means_2d.retain_grad()

    covariance_2d = []
    for index in range(means_3d.size(0)):
        x, y, z = means_3d[index]
        zero = torch.zeros((), device=means_3d.device)
        jacobian = torch.stack(
            [
                torch.stack([fx / z, zero, -fx * x / z.square()]),
                torch.stack([zero, fy / z, -fy * y / z.square()]),
            ]
        )
        projected = (
            jacobian
            @ covariance_values[index]
            @ jacobian.transpose(0, 1)
        )
        projected = projected + 1e-4 * torch.eye(
            2,
            device=means_3d.device,
        )
        covariance_2d.append(projected)

    return means_2d, torch.stack(covariance_2d), depth


def render_gaussians(
    means_3d,
    log_scales,
    quaternion,
    opacity_logits,
    sh_coefficients,
    height=8,
    width=8,
):
    covariances = covariance_3d(log_scales, quaternion)
    means_2d, covariances_2d, depth = project_gaussians(
        means_3d,
        covariances,
        height,
        width,
    )

    y_grid, x_grid = torch.meshgrid(
        torch.arange(height, device=means_3d.device, dtype=means_3d.dtype),
        torch.arange(width, device=means_3d.device, dtype=means_3d.dtype),
        indexing="ij",
    )
    pixels = torch.stack([x_grid, y_grid], dim=-1)

    colors = evaluate_sh(sh_coefficients, -means_3d)
    opacities = torch.sigmoid(opacity_logits).squeeze(-1)

    image = torch.zeros(height, width, 3, device=means_3d.device)
    transmittance = torch.ones(height, width, device=means_3d.device)

    for gaussian_index in depth.argsort():
        offset = pixels - means_2d[gaussian_index]
        inverse_covariance = torch.linalg.inv(
            covariances_2d[gaussian_index]
        )
        mahalanobis = torch.einsum(
            "...i,ij,...j->...",
            offset,
            inverse_covariance,
            offset,
        )
        gaussian = torch.exp(-0.5 * mahalanobis)
        alpha = (
            opacities[gaussian_index] * gaussian
        ).clamp(0.0, 0.99)
        weight = transmittance * alpha
        image = image + weight[..., None] * colors[gaussian_index]
        transmittance = transmittance * (1 - alpha)

    info = {
        "means_2d": means_2d,
        "depth": depth,
        "scales": torch.exp(log_scales),
        "opacity": opacities,
    }
    return image, info

## 5. Adaptive Density Control: clone, split, prune, opacity reset

High image-plane gradient + small scale → clone. High gradient + large scale → split. Low opacity → prune. Opacity can be periodically reset. The thresholds are deliberately high/low for a small deterministic demonstration; the operations themselves are the standard 3DGS density-control branches.

In [ ]:
@torch.no_grad()
def adaptive_density_control(
    means,
    log_scales,
    quaternion,
    opacity_logits,
    sh_coefficients,
    projected_gradient,
    gradient_threshold,
    scale_threshold,
    opacity_threshold,
):
    scales = torch.exp(log_scales).max(dim=-1).values
    high_gradient = projected_gradient >= gradient_threshold

    clone_mask = high_gradient & (scales <= scale_threshold)
    split_mask = high_gradient & (scales > scale_threshold)
    keep_mask = torch.sigmoid(opacity_logits).squeeze(-1) >= opacity_threshold
    # Split replaces the large parent with smaller children; cloning keeps its parent.
    original_keep_mask = keep_mask & ~split_mask

    new_means = [means[original_keep_mask]]
    new_log_scales = [log_scales[original_keep_mask]]
    new_quaternion = [quaternion[original_keep_mask]]
    new_opacity = [opacity_logits[original_keep_mask]]
    new_sh = [sh_coefficients[original_keep_mask]]

    if clone_mask.any():
        new_means.append(means[clone_mask].clone())
        new_log_scales.append(log_scales[clone_mask].clone())
        new_quaternion.append(quaternion[clone_mask].clone())
        new_opacity.append(opacity_logits[clone_mask].clone())
        new_sh.append(sh_coefficients[clone_mask].clone())

    if split_mask.any():
        parent_means = means[split_mask]
        parent_scales = torch.exp(log_scales[split_mask])
        parent_rotation = quaternion_to_rotation(quaternion[split_mask])

        for sign in (-1.0, 1.0):
            local_offset = torch.zeros_like(parent_means)
            local_offset[:, 0] = sign * 0.5 * parent_scales[:, 0]
            world_offset = torch.einsum(
                "nij,nj->ni",
                parent_rotation,
                local_offset,
            )
            new_means.append(parent_means + world_offset)
            new_log_scales.append(
                torch.log(parent_scales / 1.6)
            )
            new_quaternion.append(quaternion[split_mask].clone())
            new_opacity.append(opacity_logits[split_mask].clone())
            new_sh.append(sh_coefficients[split_mask].clone())

    result = {
        "means": torch.cat(new_means, dim=0),
        "log_scales": torch.cat(new_log_scales, dim=0),
        "quaternion": torch.cat(new_quaternion, dim=0),
        "opacity_logits": torch.cat(new_opacity, dim=0),
        "sh": torch.cat(new_sh, dim=0),
        "cloned": int(clone_mask.sum().item()),
        "split": int(split_mask.sum().item()),
        "pruned": int((~keep_mask).sum().item()),
    }
    return result


@torch.no_grad()
def reset_opacity(opacity_logits, target_opacity=0.01):
    target_logit = math.log(target_opacity / (1 - target_opacity))
    opacity_logits.clamp_(max=target_logit)


means = nn.Parameter(
    torch.tensor(
        [
            [-0.12, -0.08, 1.8],
            [0.15, -0.05, 2.1],
            [-0.05, 0.15, 2.4],
        ],
        device=device,
    )
)
log_scales = nn.Parameter(
    torch.log(
        torch.tensor(
            [
                [0.05, 0.05, 0.05],
                [0.15, 0.12, 0.10],
                [0.08, 0.08, 0.08],
            ],
            device=device,
        )
    )
)
quaternion = nn.Parameter(
    F.normalize(torch.randn(3, 4, device=device), dim=-1)
)
opacity_logits = nn.Parameter(
    torch.tensor([[1.0], [0.8], [-8.0]], device=device)
)
sh = nn.Parameter(torch.randn(3, 4, 3, device=device) * 0.2)

rendered, render_info = render_gaussians(
    means,
    log_scales,
    quaternion,
    opacity_logits,
    sh,
)
target_image = torch.zeros_like(rendered)
reconstruction_loss = F.l1_loss(rendered, target_image)
reconstruction_loss.backward()

projected_gradient = render_info["means_2d"].grad.norm(dim=-1)
# Force both high-gradient branches to be testable without replacing the criterion.
threshold = projected_gradient.min().item() - 1e-8
refined = adaptive_density_control(
    means.detach(),
    log_scales.detach(),
    quaternion.detach(),
    opacity_logits.detach(),
    sh.detach(),
    projected_gradient.detach(),
    gradient_threshold=threshold,
    scale_threshold=0.10,
    opacity_threshold=0.005,
)

assert refined["cloned"] >= 1
assert refined["split"] >= 1
assert refined["pruned"] >= 1

reset_test = opacity_logits.detach().clone()
reset_opacity(reset_test, target_opacity=0.01)

print("clone/split/prune:",
      refined["cloned"], refined["split"], refined["pruned"])
print("Gaussian count before/after:", means.size(0), refined["means"].size(0))

## Structural checklist

NeRF assertions check 8 position layers, the layer-4 skip, separate density/color branches, and both coarse/fine rendering passes. 3DGS executes quaternion anisotropy, SH color, projection/compositing, plus all adaptive density-control branches rather than stopping at a fixed-Gaussian renderer.